# Auditoría documental con RAG: del documento al dictamen

**Taller práctico · Módulos 4 y 5 · Duración: 105 minutos**

## El caso que vas a resolver

Un equipo de operaciones digitales necesita revisar sus documentos de seguridad y atención de incidentes. Quiere responder preguntas como: **¿cuánto puede permanecer inactiva una consola?, ¿qué versión de una guía corresponde a un incidente?, ¿una excepción cuenta con autorización documentada?**

El expediente contiene políticas, estándares, manuales y registros con fechas y versiones diferentes. Tu trabajo será **encontrar los párrafos adecuados, contrastar sus afirmaciones y entregar un dictamen que permita localizar cada fuente**.

Usaremos una forma de trabajo basada en **RAG (*Retrieval-Augmented Generation*, generación aumentada por recuperación)**: primero buscamos información pertinente y después la incorporamos como contexto para producir y revisar una respuesta. El notebook gestiona la recuperación, las verificaciones y las evidencias; **Microsoft 365 Copilot Chat Básico** se utiliza para redactar y contrastar una respuesta a partir del contexto seleccionado.

**Recorrido de principio a fin:**

**1. Recuperar fuentes → 2. Preparar el contexto para generar → 3. Comprobar los hallazgos → 4. Medir y registrar → 5. Emitir el dictamen.**

Trabajaremos primero con el **caso A01** (duración de sesiones), ampliaremos el análisis a otros documentos y cerraremos con el **caso A07**, que integra los hallazgos del expediente.

**Cómo trabajar:** ejecuta cada celda con **Shift+Enter**, modifica solo los valores marcados **MODIFIQUE AQUÍ** y completa tus respuestas debajo de cada reto. Guarda con **Ctrl+S**.

## 0. Preparar el entorno y conocer el expediente — 5 min

### 0A. Bibliotecas del laboratorio

Abre **`Capitulo01/materiales`** en Visual Studio Code y, dentro de esa carpeta, abre este notebook. Selecciona el kernel de Python del curso.

El archivo `requirements.txt` reúne las bibliotecas de lectura de tablas, operaciones matemáticas y recuperación. La celda siguiente las comprueba e instala únicamente cuando falta un paquete o su versión requiere un ajuste para cumplir los requisitos.

In [ ]:
# Instala las dependencias declaradas en el mismo entorno del kernel.
# Las versiones compatibles ya instaladas se conservan.
%pip install -r requirements.txt --progress-bar on

### 0B. ¿Qué información contienen los archivos?

La carpeta **`documentos/`** tiene ocho archivos `.md`. Cada uno representa un documento controlado del caso de operaciones digitales:

| Documento | Información que contiene |
|---|---|
| [`POL-SEG-014` v3.2](documentos/POL-SEG-014_v3.2.md) | Política de autenticación, segundo factor y autorización de excepciones. |
| [`EST-SEG-009` v2.1](documentos/EST-SEG-009_v2.1.md) | Estándar de seguridad que establece el tiempo de inactividad de sesiones privilegiadas. |
| [`MAN-OPS-011` v2.3](documentos/MAN-OPS-011_v2.3.md) | Manual de operación con el parámetro aplicado a la consola de canales. |
| [`PRC-INC-005` v3.1](documentos/PRC-INC-005_v3.1.md) | Procedimiento de atención y tiempos de escalamiento de incidentes críticos. |
| [`GUI-INC-004` v1.8](documentos/GUI-INC-004_v1.8.md) | Guía utilizada antes del cambio de septiembre de 2026. |
| [`GUI-INC-004` v2.0](documentos/GUI-INC-004_v2.0.md) | Guía vigente desde septiembre de 2026. |
| [`REG-EXC-006` v1.0](documentos/REG-EXC-006_v1.0.md) | Extracto del registro de solicitudes y aprobaciones de excepción. |
| [`NOT-OPS-018` v1.1](documentos/NOT-OPS-018_v1.1.md) | Nota de transferencia operativa que incluye un texto que debe analizarse con precaución. |

Puedes abrir cualquiera de los documentos desde su enlace para revisar la fuente completa. Los otros archivos apoyan el ejercicio:

- **`catalogo_documental.csv`:** relaciona cada archivo con su código, versión y vigencia.
- **`casos_auditoria.csv`:** contiene las siete preguntas que investigaremos (A01 a A07).
- **`plan_consultas.csv`:** propone subconsultas para localizar varias piezas de evidencia.
- **`respuestas_de_contraste.jsonl`:** incluye respuestas de ejemplo que compararemos con las fuentes.
- **`anexo_parametros_sesion.csv`:** registra valores observados de configuración por componente.
- **`motor_rag.py`:** contiene las funciones Python que leen, buscan y comprueban el expediente.
- **`manifiesto_materiales.json`:** permite verificar que los documentos utilizados conservan su contenido.

Los párrafos que podemos citar tienen identificadores como **`[EST-SEG-009-v2.1#2.2]`**: código del documento, versión y apartado. En el dictamen, ese identificador permitirá encontrar el texto que respalda una observación.

**En la siguiente celda** cargaremos el expediente y veremos tres salidas: el catálogo de versiones, los casos disponibles y dos párrafos relacionados con la duración de las sesiones.

In [ ]:
# Herramientas para archivos, tablas, gráficos y registro de la ejecución.
from pathlib import Path
from importlib.metadata import version
import json, sys, uuid
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Conservamos las ubicaciones relativas originales de los materiales.
MATERIALES = next((c.resolve() for base in [Path.cwd(), *Path.cwd().parents]
                   for ruta in ['.', 'materiales', 'Capitulo01/materiales']
                   if (c := base / ruta).joinpath('manifiesto_materiales.json').is_file()), None)
if MATERIALES is None:
    raise FileNotFoundError('Abra Capitulo01/materiales en Visual Studio Code.')
sys.path.insert(0, str(MATERIALES))
from motor_rag import (MotorRAG, prompt_aumentado, hallazgo_controlado,
                       evaluacion_estructura, puerta_revision, traza_segura,
                       fecha_aplicable, huella, datos_serie)

SALIDAS = MATERIALES / 'resultados'
SALIDAS.mkdir(exist_ok=True)
motor = MotorRAG(MATERIALES, peso_lexico=0.60)  # 60 % BM25; 40 % LSA.
print(f'Expediente cargado: {len(motor.catalogo)} documentos, {len(motor.fragmentos)} párrafos citables y {len(motor.casos)} casos.')
print('Tabla 1. Catálogo: código, versión y fechas que determinan la vigencia de cada documento.')
display(motor.catalogo[['codigo', 'version', 'vigencia_desde', 'reemplazado_desde']])
print('Tabla 2. Casos que se investigarán: cada código corresponde a una pregunta del expediente.')
display(motor.casos[['caso_id', 'nombre']])
# Leemos dos párrafos directamente del expediente para reconocer cómo se cita una fuente.
referencias_ejemplo = ['EST-SEG-009-v2.1#2.2', 'MAN-OPS-011-v2.3#3.1']
print('Tabla 3. Párrafos originales sobre sesiones: observa la cifra indicada en cada fuente.')
with pd.option_context('display.max_colwidth', 280):
    display(motor.fragmentos.loc[motor.fragmentos.cita.isin(referencias_ejemplo), ['cita', 'texto']])

**Cómo leer lo que acabas de ver:** 
- **tabla 1**, `vigencia_desde` indica desde cuándo aplica una versión y `reemplazado_desde` cuándo deja de aplicarse.
- **tabla 2** presenta los casos de auditoría. 
- **tabla 3** muestra dos fragmentos del expediente: el estándar fija **15 minutos** y el manual indica **30 minutos**. 
Esta diferencia será nuestra primera investigación.

---

## Reto 1. Recuperar los documentos correctos — 10 min
**Etapa del ciclo: 1 · Recuperar contexto**

### Contexto del caso de uso

La pregunta **A01** es si los documentos coinciden en el tiempo de inactividad de una consola privilegiada. Para responder, necesitamos localizar **ambos** párrafos, con sus códigos y versiones, antes de elaborar una conclusión.

El componente de recuperación de `motor_rag.py` combina dos métodos:

- **BM25:** da prioridad a fragmentos que coinciden con palabras de la búsqueda, como *sesión*, *inactividad* o *consola*.
- **LSA:** identifica asociaciones estadísticas entre palabras de los documentos. Se construye con TF-IDF y reducción de dimensiones; proporciona una segunda puntuación para ordenar los textos.

Una búsqueda que combina ambos puntajes se denomina **recuperación híbrida**. Aquí LSA representa los textos mediante vectores calculados a partir del expediente. Un **modelo neuronal de embeddings** utilizaría representaciones aprendidas previamente; sería otra manera de calcular semejanza semántica.

Compararemos **una pregunta general** con **varias preguntas por subtema**. `TOP_K` define cuántos candidatos examina cada búsqueda y `PRESUPUESTO` limita cuántos caracteres de texto se incorporan al contexto final.

In [ ]:
# Buscamos evidencia sobre la duración de las sesiones.
CASO, FECHA_CORTE = 'A01', '2026-10-10'
TOP_K = 5  # MODIFIQUE AQUÍ: compare 3 y 7.
PRESUPUESTO = 5600  # MODIFIQUE AQUÍ: compare 3200.

# La búsqueda guiada consulta por separado los aspectos del caso.
rec_simple = motor.recuperar_caso(CASO, TOP_K, PRESUPUESTO, FECHA_CORTE)
rec_guiada = motor.recuperar_caso(CASO, TOP_K, PRESUPUESTO, FECHA_CORTE, descomponer=True)

# La cobertura compara las citas recuperadas contra referencias de evaluación.
comparacion = pd.DataFrame([
    {'búsqueda': nombre, 'fragmentos': len(r.seleccion),
     'cobertura': motor.cobertura(CASO, r.seleccion)['cobertura']}
    for nombre, r in [('General', rec_simple), ('Por subtemas', rec_guiada)]
])
print('Tabla 1. Cobertura: proporción de referencias necesarias que encontró cada estrategia.')
display(comparacion)
# Incluimos el texto recuperado para entender por qué aparece cada referencia.
vista_recuperacion = rec_guiada.seleccion[['cita', 'texto', 'bm25', 'lsa', 'estado_corte']].copy()
vista_recuperacion['texto'] = vista_recuperacion['texto'].str.slice(0, 170)
print('Tabla 2. Fragmentos recuperados: identifica las cifras, las fuentes y la vigencia.')
with pd.option_context('display.max_colwidth', 180):
    display(vista_recuperacion.round(3))

### Cómo leer la recuperación

En la **primera tabla**, `fragmentos` es la cantidad seleccionada y `cobertura` va de **0 a 1**. Una cobertura de `1.0` indica que la búsqueda recuperó todas las referencias esperadas para **A01**, según la lista de evaluación del caso. En la **segunda tabla**, `cita` permite volver al párrafo original y `texto` muestra el inicio de lo recuperado. Las columnas `bm25` y `lsa` son puntuaciones de métodos distintos; sirven para comprender cómo se ordenan los fragmentos, no como porcentajes de certeza.

**Comprueba:** ¿aparecen los párrafos `EST-SEG-009-v2.1#2.2` y `MAN-OPS-011-v2.3#3.1`? Modifica `TOP_K` y vuelve a ejecutar este reto para observar qué cambia.

**Interpretación 1.** ¿Qué estrategia recuperó mejor las dos fuentes y cómo lo comprobaste en las tablas?

Respuesta: ...

---

## Reto 2. Preparar el contexto para obtener una respuesta fundamentada — 15 min
**Etapa del ciclo: 2 · Preparar la generación**

### Contexto del caso de uso

Ya encontramos textos relacionados con el problema. Ahora los presentaremos junto con la pregunta y las reglas de la revisión. Esa combinación se denomina **prompt aumentado**: contiene **la tarea, la evidencia, las referencias y el formato de respuesta solicitado**.

Compararemos dos prompts construidos sobre las **mismas fuentes**. El primero pide un resumen. El segundo solicita identificar las discrepancias, citar cada cifra, distinguir versiones y explicar qué debe verificar un responsable. Una respuesta así permite revisar si la conclusión está sustentada.

`motor.fila_caso()` recupera la pregunta de A01 y `prompt_aumentado()` coloca el contexto y las instrucciones en un texto que puede enviarse a Copilot. La celda mostrará el tamaño de ambos prompts y las **instrucciones** del más completo. Guardará el texto íntegro, incluidos los fragmentos, en `resultados/paquete_contexto_A01.md`.

In [ ]:
# fila_caso aporta la pregunta; prompt_aumentado agrega evidencia y reglas.
ficha = motor.fila_caso(CASO)
prompt_base = prompt_aumentado(ficha, rec_guiada, riguroso=False)
prompt_fundamentado = prompt_aumentado(ficha, rec_guiada, riguroso=True)

# Conservamos el nombre original del archivo para el ejercicio de Copilot.
(SALIDAS / 'paquete_contexto_A01.md').write_text(
    '# Contexto aumentado A01\n\n```text\n' + prompt_fundamentado + '\n```\n', encoding='utf-8')
print('Comparación de tamaños: los caracteres adicionales corresponden a instrucciones y estructura de la respuesta.')
print('Prompt de resumen:', len(prompt_base), 'caracteres')
print('Prompt fundamentado:', len(prompt_fundamentado), 'caracteres')
print('Instrucciones de revisión que acompañarán a las fuentes:')
print(prompt_fundamentado.split('--- INICIO DE FUENTES')[0])
print('Archivo preparado para Copilot: resultados/paquete_contexto_A01.md')

### Leer el prompt y probarlo en Copilot

La salida muestra **qué se pide al modelo** antes de leer los fragmentos. Abre `resultados/paquete_contexto_A01.md` y copia el contenido entre las marcas de código a un chat nuevo de **Microsoft 365 Copilot Chat Básico**. Las fuentes siguen identificadas por su código y apartado.

Al revisar la respuesta, busca estos tres elementos: **los 15 minutos del estándar**, **los 30 minutos del manual** y **una observación sustentada en ambas referencias**. El objetivo es comprobar cómo utiliza Copilot la evidencia para redactar la conclusión.

**Concepto relacionado con la generación:** un LLM predice tokens para construir texto. **Temperatura** y **`top_p`** son estrategias de selección de tokens disponibles en ciertos modelos; influyen en la variación de la redacción. En este ejercicio trabajamos sobre los elementos directamente comprobables: la pregunta, los fragmentos, las citas y la instrucción de justificar cada afirmación.

**Interpretación 2.** ¿Qué información añade el prompt aumentado a la pregunta original? ¿Qué dos referencias utilizó Copilot para justificar su respuesta?

Respuesta: ...

---

## Reto 3. Comprobar si las afirmaciones están respaldadas — 15 min
**Etapa del ciclo: 3 · Evaluar evidencia y respuesta**

### Contexto del caso de uso

Una conclusión puede citar un documento y, aun así, interpretar mal su contenido. Por ejemplo, afirmar **«el estándar permite 45 minutos»** contradice el párrafo que acabamos de leer. La revisión necesita comprobar **dos cosas diferentes**:

1. **Referencia:** que la cita existe, corresponde a una versión aplicable y figura entre los fragmentos recuperados.
2. **Contenido:** que el párrafo citado efectivamente respalda el dato de la afirmación.

`respuestas_de_contraste.jsonl` contiene respuestas preparadas para A01 y otros casos. Algunas omiten información o referencias; otras incluyen el contexto necesario. Primero verás dos textos correspondientes a A01 y después una tabla que indica cuántas citas encontró el comprobador y qué proporción pertenece al expediente y al contexto recuperado.

In [ ]:
# Revisamos los identificadores y la vigencia de respuestas de contraste.
respuestas = datos_serie(MATERIALES / 'respuestas_de_contraste.jsonl')
revisiones = []
for r in respuestas.to_dict('records'):
    contexto = motor.recuperar_caso(r['caso_id'], 7, 8000, FECHA_CORTE, True)
    control = motor.comprobacion_citas(r['texto'], contexto.seleccion, FECHA_CORTE)
    revisiones.append({'caso': r['caso_id'], 'variante': r['variante'],
                       'citas': control['total'], 'validas': control['tasa_validas'],
                       'en_contexto': control['tasa_en_contexto']})
print('Ejemplos A01: compara qué afirma cada respuesta sobre los plazos de sesión.')
display(respuestas.loc[respuestas.caso_id == 'A01', ['variante', 'texto']])
print('Control automático: número de citas, proporción de citas existentes y citas presentes en el contexto.')
display(pd.DataFrame(revisiones))

**Cómo leer las salidas:** `citas` es el número de referencias incluidas en la respuesta. `validas = 1.0` significa que todas esas citas existen en el expediente; `en_contexto = 1.0` indica que todas estaban entre los fragmentos seleccionados para la consulta. **Estos indicadores validan referencias**, mientras que la lectura del texto permite validar las cifras y las conclusiones.

### Resolver dos situaciones documentales

Ahora aplicaremos `hallazgo_controlado()`, una función que **extrae y compara información específica de los párrafos recuperados** siguiendo reglas del caso. Analizaremos:

- **A01:** el estándar y el manual contienen tiempos distintos.
- **A05:** el extracto sobre **EXC-227** permite identificar qué información adicional hace falta para verificar su aprobación.

La tabla mostrará qué observación produce el componente Python, sus referencias y su estado. Después compararemos una afirmación con el texto original del estándar.

In [ ]:
# Los hallazgos extractivos se apoyan en las fuentes realmente recuperadas.
rec_A01 = rec_guiada
rec_A05 = motor.recuperar_caso('A05', 5, 5600, FECHA_CORTE, True)
hall_A01 = hallazgo_controlado(motor, 'A01', rec_A01)
hall_A05 = hallazgo_controlado(motor, 'A05', rec_A05)
print('Tabla de hallazgos: tipo de situación, observación sustentada, fuentes y estado de revisión.')
display(pd.DataFrame([{'caso': 'A01', **hall_A01}, {'caso': 'A05', **hall_A05}])
        [['caso', 'tipo', 'observacion', 'citas', 'estado']])

# Una cita válida también necesita respaldar el contenido de la frase.
ficha_estandar = motor.ficha_cita('EST-SEG-009-v2.1#2.2', rec_A01.seleccion)
print('Comprobación del contenido: contrasta la afirmación siguiente con el párrafo del estándar.')
print('Afirmación para revisar: «El estándar permite 45 minutos»')
print('Evidencia recuperada:', ficha_estandar['texto'] if ficha_estandar else
      'Amplíe TOP_K para incorporar el fragmento del estándar al contexto.')

**Cómo interpretar el resultado:** en A01, el campo `observacion` compara las dos cifras recuperadas; en A05, el campo `estado` refleja que la aprobación requiere evidencia documental adicional. La impresión final muestra el **texto del estándar** que permite aceptar o rechazar la afirmación de 45 minutos.

**Interpretación 3.** ¿Por qué una cita existente puede acompañar una afirmación incorrecta? ¿Qué diferencia observas entre señalar una contradicción y solicitar evidencia de una aprobación?

Respuesta: ...

---

## Reto 4. Volver a buscar cuando la evidencia necesita completarse — 10 min
**Regreso a la etapa 1 · Recuperar y volver a evaluar**

### Contexto del caso de uso

La investigación avanza del caso A01 al **A03**. El equipo debe determinar qué guía de escalamiento se aplicaba a un incidente crítico clasificado el **15 de septiembre de 2026**. Hay dos versiones de `GUI-INC-004`: la versión 1.8 corresponde al periodo anterior a septiembre y la 2.0 entra en vigor el 1 de septiembre.

El componente realizará una **reconsulta guiada**: intentará recuperar evidencia con `top_k` creciente (`3`, `5`, `7`) y comprobará si las referencias obtenidas permiten formular el hallazgo. Se detendrá cuando encuentre información suficiente. Esta secuencia —**evaluar → ampliar consulta → volver a evaluar**— ayuda a comprender los flujos RAG con decisiones por etapas.

In [ ]:
# A03 necesita una guía aplicable al incidente del 15 de septiembre.
pasos = []
for k in [3, 5, 7]:
    rec = motor.recuperar_caso('A03', k, 6000, FECHA_CORTE, True)
    hall = hallazgo_controlado(motor, 'A03', rec, fecha_evento='2026-09-15')
    pasos.append({'top_k': k, 'estado': hall['estado'], 'citas': hall['citas']})
    if hall['estado'] != 'Abstencion':  # La evidencia define cuándo detenerse.
        break
print('Tabla de reconsultas: top_k muestra cuántos candidatos se buscaron y el estado indica si hay evidencia suficiente.')
display(pd.DataFrame(pasos))

# Consultamos los metadatos para distinguir versiones según el día del evento.
guias = motor.catalogo.query("codigo == 'GUI-INC-004'")
print('Comprobación histórica: versión aplicable antes y después del cambio del 01/09/2026.')
for fecha in ['2026-08-15', '2026-09-15']:
    vigentes = guias.loc[guias.apply(lambda fila: fecha_aplicable(fila, fecha), axis=1)]
    print(fecha, '→ versión aplicable:', vigentes.version.tolist())

**Cómo leer la salida:** la tabla enseña cuántos intentos necesitó el análisis; `Abstencion` indica que el contexto recuperado requiere ampliarse antes de concluir. Debajo aparecen las versiones que aplican en **agosto** y **septiembre**. La fecha del incidente determina qué plazo utilizar, incluso si una búsqueda devuelve una versión anterior.

**Interpretación 4.** ¿Qué versión corresponde al evento del 15 de septiembre y qué evidencia respalda esa decisión? ¿Cuándo convendría realizar otra búsqueda antes de responder?

Respuesta: ...

---

## Reto 5. Evaluar los hallazgos de todo el expediente — 20 min
**Etapa del ciclo: 3 · Evaluación de calidad**

### Contexto del caso de uso

Hasta aquí investigamos casos individuales. Ahora repetiremos la recuperación y la revisión sobre las **siete preguntas del expediente** para conocer cómo funciona el procedimiento de manera consistente.

Utilizaremos dos medidas:

- **Cobertura:** proporción de referencias esperadas que llegaron al contexto. Si un caso necesita dos fragmentos y recuperamos uno, su cobertura es `0.5`.
- **Citas en contexto:** proporción de las referencias emitidas en el hallazgo que efectivamente estuvieron en los textos seleccionados.

`TOP_K_EVAL` permite comparar la calidad de recuperación al cambiar la cantidad de fragmentos buscados. La primera salida será **una fila por caso**. Las tablas se guardarán para consultarlas durante la revisión final.

In [ ]:
# Ejecutamos los siete casos con una misma configuración de recuperación.
TOP_K_EVAL = 5  # MODIFIQUE AQUÍ: compare 3 y 7.
PRESUPUESTO_EVAL = 5600
recuperaciones_eval, hallazgos_eval = {}, {}
filas_eval, filas_fuentes = [], []
for caso in motor.casos.caso_id:
    rec = motor.recuperar_caso(caso, TOP_K_EVAL, PRESUPUESTO_EVAL, FECHA_CORTE, True)
    hall = hallazgo_controlado(motor, caso, rec)
    recuperaciones_eval[caso], hallazgos_eval[caso] = rec, hall
    filas_eval.append(evaluacion_estructura(motor, caso, rec, hall))
    filas_fuentes += [{'caso_id': caso, 'cita': f.cita, 'documento': f.archivo,
                       'puntaje': round(f.puntaje, 4), 'estado_corte': f.estado_corte}
                      for f in rec.seleccion.itertuples()]

# Guardamos los resultados para poder contrastar otra configuración.
evaluacion_hallazgos = pd.DataFrame(filas_eval)
fuentes_recuperadas = pd.DataFrame(filas_fuentes)
print('Evaluación de siete casos: cobertura de fuentes, citas respaldadas por contexto y estado de cada hallazgo.')
display(evaluacion_hallazgos[['caso_id', 'cobertura', 'tasa_citas_en_contexto', 'estado_hallazgo']])
evaluacion_hallazgos.to_csv(SALIDAS / 'evaluacion_hallazgos.csv', index=False)
fuentes_recuperadas.to_csv(SALIDAS / 'fuentes_recuperadas.csv', index=False)

**Cómo leer la tabla:** cada `caso_id` coincide con una pregunta del expediente que conocimos al inicio. Una cobertura más alta significa que se recuperaron más referencias previstas para el caso. La columna `estado_hallazgo` informa si el procedimiento pudo formular una observación o documentó la necesidad de ampliar la evidencia.

### Evaluación humana: ¿el hallazgo se puede utilizar?

Los indicadores anteriores permiten revisar fuentes y referencias. Una **rúbrica humana** agrega criterios sobre si la conclusión es útil y prudente. Califica de `0` a `2` cinco aspectos: fundamentación, vigencia, trazabilidad, manejo de incertidumbre y utilidad. El máximo es **10 puntos** por caso.

Las puntuaciones de A01 y A05 son un punto de partida para el ejercicio: **MODIFIQUE AQUÍ** las que consideres necesarias después de comparar cada observación con los documentos.

In [ ]:
# MODIFIQUE AQUÍ: 0 = insuficiente; 1 = parcial; 2 = cumple.
REVISION_HUMANA = {
    'A01': {'fundamentacion': 2, 'vigencia': 1, 'trazabilidad': 2,
            'incertidumbre': 2, 'utilidad': 1},
    'A05': {'fundamentacion': 2, 'vigencia': 1, 'trazabilidad': 2,
            'incertidumbre': 2, 'utilidad': 1},
}
# Sumamos los cinco criterios de cada caso (máximo 10 puntos).
rubrica = pd.DataFrame([{'caso_id': caso, **criterios, 'total_10': sum(criterios.values())}
                        for caso, criterios in REVISION_HUMANA.items()])
assert rubrica.filter(items=list(REVISION_HUMANA['A01'])).isin([0, 1, 2]).all().all()
print('Rúbrica humana: cada columna aporta de 0 a 2 puntos; total_10 resume los cinco criterios.')
display(rubrica)
rubrica.to_csv(SALIDAS / 'rubrica_revision.csv', index=False)

**Cómo interpretar la rúbrica:** un puntaje alto significa que **la revisión realizada** considera mejor sustentado y utilizable el hallazgo según esos criterios. La puntuación responde a tu lectura; sirve para documentar decisiones de calidad y discutir diferencias de criterio.

**Interpretación 5.** ¿Qué caso requiere mejorar la cobertura? ¿Qué criterio de la rúbrica cambiarías después de verificar los documentos y por qué?

Respuesta: ...

---

## Reto 6. Medir el funcionamiento del recuperador — 15 min
**Etapa del ciclo: 4 · Registrar la operación**

### Contexto del caso de uso

Un sistema utilizado de manera recurrente necesita indicadores para comparar configuraciones y explicar sus resultados. Probaremos cuatro combinaciones de **cantidad de fragmentos** (`top_k`) y **presupuesto de texto**.

La tabla muestra, para cada configuración, tres observaciones principales:

- **Cobertura media:** qué proporción de referencias esperadas llega al contexto en los siete casos.
- **Caracteres promedio:** cuánto texto se entrega para formar el prompt.
- **Latencia en milisegundos:** cuánto tardó realmente la búsqueda local.

También calcularemos un **costo ilustrativo** aplicando una tarifa de ejemplo a una estimación de tokens. Esta columna permite estudiar el compromiso entre volumen de contexto y costo de generación.

In [ ]:
# Comparamos cuatro presupuestos de recuperación para los mismos siete casos.
filas_bench = []
for k, presupuesto in [(3, 3200), (3, 5600), (5, 3200), (5, 5600)]:
    for caso in motor.casos.caso_id:
        rec = motor.recuperar_caso(caso, k, presupuesto, FECHA_CORTE, True)
        filas_bench.append({'caso': caso, 'top_k': k, 'presupuesto': presupuesto,
                            'cobertura': motor.cobertura(caso, rec.seleccion)['cobertura'],
                            'caracteres': len(rec.contexto), 'latencia_ms': rec.latencia_ms})

# Los promedios permiten comparar calidad, volumen y tiempo local.
benchmark_detalle = pd.DataFrame(filas_bench)
benchmark = benchmark_detalle.groupby(['top_k', 'presupuesto'], as_index=False).mean(numeric_only=True)
benchmark['costo_ilustrativo_usd'] = benchmark.caracteres / 4 / 1_000_000 * 2
benchmark_detalle.to_csv(SALIDAS / 'benchmark_configuraciones.csv', index=False)
print('Comparación de configuraciones: cobertura y latencia medidas; costo ilustrativo estimado a partir de caracteres.')
display(benchmark.round(4))

# Representamos cómo cambia la cobertura al aumentar los caracteres recuperados.
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for k, grupo in benchmark.groupby('top_k'):
    ax.plot(grupo.caracteres, grupo.cobertura, marker='o', label=f'top_k={k}')
ax.set(xlabel='Caracteres promedio', ylabel='Cobertura media', title='Contexto y cobertura', ylim=(0, 1.05))
ax.legend(); fig.tight_layout()
fig.savefig(SALIDAS / 'calidad_vs_contexto.png', dpi=140)
print('Gráfico: eje horizontal = tamaño del contexto; eje vertical = cobertura media. Compara cada curva.')
plt.show()

**Cómo leer las salidas:** cada fila de la tabla combina `top_k` y `presupuesto`. El gráfico muestra si recuperar más texto ayuda a incorporar las fuentes necesarias. Un valor alto de cobertura con menos caracteres puede resultar más eficiente. La latencia corresponde únicamente al proceso **local de recuperación**; la estimación económica es una referencia didáctica para analizar escenarios.

### Registrar qué sucedió en cada consulta

Una **traza** es un resumen operativo que permite reconstruir una ejecución. El archivo `trazas_operacion.jsonl` guardará, por caso, identificadores, número de fragmentos recuperados y tiempo de búsqueda. La tabla presentará esos tres campos para facilitar la revisión.

In [ ]:
# Generamos un identificador y guardamos una traza por caso evaluado.
EJECUCION_ID = 'rag-' + uuid.uuid4().hex[:12]
trazas = [traza_segura(caso, rec, EJECUCION_ID, 'evaluacion_recuperacion')
          for caso, rec in recuperaciones_eval.items()]
(SALIDAS / 'trazas_operacion.jsonl').write_text(
    ''.join(json.dumps(t, ensure_ascii=False) + '\n' for t in trazas), encoding='utf-8')
print('Registro operativo: caso, cantidad de fragmentos y duración medida de la recuperación.')
display(pd.DataFrame(trazas)[['caso_id', 'n_recuperados', 'latencia_recuperacion_ms']])

**Cómo leer la traza:** permite saber qué caso se consultó, cuántos fragmentos se recuperaron y cuánto tiempo tomó la búsqueda. Al conservar también el identificador de ejecución, podremos relacionar estas medidas con el dictamen.

### Una decisión adicional al llevar RAG a producción: memoria del modelo

El componente generativo suele ejecutarse sobre infraestructura que reserva memoria para sus pesos. La **cuantización** reduce la cantidad de bits utilizados por esos pesos. En el ejemplo siguiente se estima **solo la memoria de pesos** de un modelo de referencia de 7.000 millones de parámetros, comparando formatos de 16, 8 y 4 bits.

**En la siguiente tabla** `bits_por_peso` expresa la precisión numérica usada para almacenar un peso y `solo_pesos_GiB` estima memoria. La comparación ayuda a explicar por qué el tamaño del modelo también influye en decisiones de operación.

In [ ]:
# Ejemplo de infraestructura: estimación matemática del espacio ocupado por los pesos.
parametros = 7_000_000_000  # Número de parámetros de referencia.
memoria = pd.DataFrame({'bits_por_peso': [16, 8, 4]})
memoria['solo_pesos_GiB'] = parametros * memoria.bits_por_peso / 8 / (1024 ** 3)
print('Memoria teórica de pesos por formato numérico, expresada en GiB.')
display(memoria.round(2))

**Interpretación 6.** ¿Qué configuración ofrece mejor cobertura con un contexto razonable? ¿Qué dato de la traza utilizarías para diagnosticar una búsqueda lenta?

Respuesta: ...

---

## Reto 7. Examinar una fuente riesgosa y un anexo tabular — 10 min
**Control transversal: seguridad y tipos de información**

### Contexto del caso de uso

Hasta ahora hemos utilizado textos que describen controles. El caso **A06** permite estudiar un riesgo distinto: la nota `NOT-OPS-018` contiene una frase dirigida al sistema de auditoría para que ignore una discrepancia. Ese contenido forma parte del documento que se analiza; la tarea del sistema sigue siendo **evaluar y citar la fuente**, con sus controles establecidos.

Primero recuperaremos esa nota y mostraremos una observación que identifica el intento de cambiar las instrucciones. La tabla incluirá el tipo de hallazgo, qué se observó y qué revisión corresponde.

In [ ]:
# A06 contiene una instrucción incrustada dentro de un documento.
rec_A06 = motor.recuperar_caso('A06', 5, 5600, FECHA_CORTE, True)
hall_A06 = hallazgo_controlado(motor, 'A06', rec_A06)
print('Análisis de A06: instrucción incrustada detectada, explicación y medida de revisión.')
display(pd.DataFrame([hall_A06])[['tipo', 'observacion', 'accion']])


**Cómo leer el resultado:** el campo `observacion` explica por qué la instrucción incrustada merece atención; `accion` muestra el control propuesto para que la evaluación permanezca basada en los documentos relevantes.

### Incorporar información de una tabla

También contamos con `anexo_parametros_sesion.csv`, que registra **tres componentes**, sus tiempos de inactividad configurados, una fecha y un identificador de revisión. Leeremos la tabla y crearemos un gráfico de barras para comparar los valores.

**Qué observarás:** la tabla conserva el dato exacto y su registro; el gráfico facilita detectar diferencias entre componentes. Esta actividad introduce la preparación de información estructurada para una recuperación que también pueda citar tablas y otros formatos documentales.

In [ ]:
# Leemos el anexo tabular y visualizamos los tiempos por componente.
anexo = pd.read_csv(MATERIALES / 'anexo_parametros_sesion.csv')
print('Anexo tabular: componente, minutos de inactividad, fecha e identificador de registro.')
display(anexo)
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar(anexo.componente, anexo.inactividad_configurada_min)
ax.set(ylabel='Minutos', title='Parámetros registrados por componente')
plt.xticks(rotation=12, ha='right'); fig.tight_layout()
fig.savefig(SALIDAS / 'anexo_visual_parametros.png', dpi=140)
print('Gráfico: compara los minutos de inactividad configurados por componente.')
plt.show()

**Cómo leer las salidas:** cada fila representa un componente distinto. Contrasta el tiempo configurado de la **consola operativa** con el estándar y el manual consultados en A01. En un sistema RAG con documentos variados, estos valores se acompañarían de una referencia al registro de origen para poder incluirlos como evidencia.

**Interpretación 7.** ¿Qué intento de manipulación aparece en A06? ¿Qué dato del anexo ayuda a revisar la diferencia detectada en A01?

Respuesta: ...

---

## Reto 8. Emitir y verificar el dictamen — 5 min
**Etapa del ciclo: 5 · Presentar un resultado trazable**

### Contexto del caso de uso

Llegó el momento de integrar las evidencias. El caso **A07** reúne hallazgos sobre duración de sesiones, vigencia de guías, excepciones y escalamiento. El resultado será un **dictamen** que pueda revisar otra persona sin repetir todo el proceso de búsqueda.

`hallazgo_controlado()` reúne observaciones basadas en los fragmentos recuperados. `puerta_revision()` comprueba la trazabilidad antes de registrar el dictamen. Veremos **una fila por asunto revisado**, su estado y los identificadores de las fuentes. El archivo `resultados/dictamen_auditoria.md` conservará las observaciones y las acciones de revisión.

In [ ]:
# Recuperamos la evidencia integrada y comprobamos que las citas son trazables.
rec_A07 = motor.recuperar_caso('A07', 5, 12000, FECHA_CORTE, True)
hall_A07 = hallazgo_controlado(motor, 'A07', rec_A07)
control_final = puerta_revision(motor, hall_A07, rec_A07)

# Una sección del dictamen por control auditado: observación, citas y acción.
lineas = ['# Dictamen de consistencia documental', '', f'Corte: {FECHA_CORTE}',
          f"Estado: {control_final['estado']}", '']
for seccion in hall_A07['secciones']:
    lineas += [f"## {seccion['subcaso']} — {seccion['tipo']}", seccion['observacion'],
               '**Fuentes:** ' + ', '.join(seccion['citas']),
               '**Límite:** ' + seccion['limite'], '**Revisión:** ' + seccion['accion'], '']
(SALIDAS / 'dictamen_auditoria.md').write_text('\n\n'.join(lineas), encoding='utf-8')
print('Secciones del dictamen A07: asunto auditado, tipo de hallazgo, estado y citas utilizadas.')
display(pd.DataFrame(hall_A07['secciones'])[['subcaso', 'tipo', 'estado', 'citas']])
print('Estado del dictamen:', control_final['estado'])

**Cómo leer el dictamen:** `subcaso` permite regresar a la pregunta original; `tipo` explica la situación; `citas` indica los párrafos que la sustentan. Abre `resultados/dictamen_auditoria.md` para encontrar también las observaciones y los pasos de revisión humana.

### Contrastar el dictamen con Copilot Chat Básico

Para cerrar el recorrido, prepararemos un segundo paquete con **la observación de A01 y sus párrafos citados**. Copia el texto de `resultados/paquete_validacion_copilot.md` a Copilot y pídele comprobar si cada afirmación está sustentada, parcialmente sustentada o carece de respaldo.

La siguiente celda guarda el archivo y muestra su ubicación para encontrarlo fácilmente.

In [ ]:
# Reunimos las fuentes de A01 con el mismo identificador usado en el expediente.
citas_para_copilot = hall_A01['citas'] or rec_A01.seleccion.cita.head(3).tolist()
fuentes_copilot = '\n'.join(f"[{c}] {motor.ficha_cita(c, rec_A01.seleccion)['texto']}"
                            for c in citas_para_copilot)
texto_copilot = (
    'Revisa el hallazgo frente a las fuentes. Clasifica cada afirmación como '
    'sustentada, parcial o sin soporte. Cita el identificador correspondiente, '
    'señala diferencias y explica qué corresponde verificar.\n\n'
    f'FUENTES:\n{fuentes_copilot}\n\nHALLAZGO:\n{hall_A01["observacion"]}'
)
(SALIDAS / 'paquete_validacion_copilot.md').write_text(texto_copilot, encoding='utf-8')
print('Paquete para revisión en Copilot: contiene el hallazgo de A01 y las fuentes que cita.')
print('Archivo:', SALIDAS / 'paquete_validacion_copilot.md')

### Comprobar que el trabajo quedó guardado

Esta última celda revisa las **referencias del dictamen**, la integridad del expediente y los archivos generados. El mensaje final indicará cuántas evidencias quedaron disponibles en la carpeta `resultados/`.

In [ ]:
# Registramos el entorno y la configuración utilizada en la evaluación.
registro = {'run_id': EJECUCION_ID, 'corte': FECHA_CORTE, 'python': sys.version.split()[0],
            'bibliotecas': {n: version(n) for n in ['numpy', 'pandas', 'scikit-learn', 'matplotlib']},
            'top_k_eval': TOP_K_EVAL, 'presupuesto_eval': PRESUPUESTO_EVAL,
            'archivos_sha256': motor.manifiesto['integridad_documentos']}
(SALIDAS / 'manifiesto_ejecucion.json').write_text(
    json.dumps(registro, ensure_ascii=False, indent=2), encoding='utf-8')

# Verificamos fuentes, citas y evidencias sin modificar ningún archivo original.
assert all(huella(MATERIALES / 'documentos' / nombre) == sha
           for nombre, sha in motor.manifiesto['integridad_documentos'].items())
assert set(hall_A07['citas']).issubset(set(rec_A07.seleccion.cita))
EVIDENCIAS = ['paquete_contexto_A01.md', 'evaluacion_hallazgos.csv', 'fuentes_recuperadas.csv',
             'rubrica_revision.csv', 'benchmark_configuraciones.csv', 'calidad_vs_contexto.png',
             'trazas_operacion.jsonl', 'anexo_visual_parametros.png', 'dictamen_auditoria.md',
             'paquete_validacion_copilot.md', 'manifiesto_ejecucion.json']
assert all((SALIDAS / nombre).is_file() for nombre in EVIDENCIAS)
print(f'Validación completada: {len(EVIDENCIAS)} evidencias y referencias verificadas.')

### Interpretación 8. Dictamen y revisión final

1. ¿Qué observaciones reúne el dictamen A07 y cómo puedes localizar los párrafos que las sustentan?
2. ¿Qué afirmación del dictamen revisarías primero con Copilot y cuál sería tu criterio de verificación?

Respuesta: ...

### Conclusiones de la práctica

1. Describe una mejora que observaste al recuperar evidencia por subtemas y preparar un prompt con referencias.
2. Explica qué comprobaciones debe realizar una persona antes de utilizar un dictamen de consistencia documental.

Respuesta: ...

### Lo que realizaste

Investigaste un expediente de operaciones digitales: localizaste sus fuentes, recuperaste párrafos mediante BM25 y LSA, preparaste prompts con contexto, examinaste afirmaciones y citas, ampliaste búsquedas, evaluaste siete casos, registraste indicadores y emitiste un dictamen trazable.

Los ejercicios de **prompts y fundamentación** se relacionan con el módulo 4 de Coursera; la **evaluación, observabilidad y seguridad** se relacionan con el módulo 5.

Guarda tu trabajo con **Ctrl+S**. Los archivos generados durante la ejecución estarán en `Capitulo01/materiales/resultados/`.
